In [1]:
!git clone https://github.com/para0107/Cluj-Road-Intelligence-System.git
%cd Cluj-Road-Intelligence-System
!pip install -r requirements.txt

Cloning into 'Cluj-Road-Intelligence-System'...
remote: Enumerating objects: 216, done.
remote: Counting objects: 100% (216/216), done.
remote: Compressing objects: 100% (168/168), done.
remote: Total 216 (delta 81), reused 163 (delta 33), pack-reused 0 (from 0)
Receiving objects: 100% (216/216), 17.96 MiB | 23.04 MiB/s, done.
Resolving deltas: 100% (81/81), done.
/kaggle/working/Cluj-Road-Intelligence-System
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 4.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.7/40.7 kB 2.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 104.9/104.9 kB 7.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 107.3/107.3 kB 6.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.6/57.6 kB 3.3 MB/s eta 0:00:00
INFO: pip is looking at multiple versions of opencv-python-headless to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━

In [2]:
import shutil
import os

# Build the base folder structure
os.makedirs('data/datasets', exist_ok=True)

print("Copying RDD2022 (This will take a few minutes for 28k images)...")
# Using the exact path you found!
shutil.copytree('/kaggle/input/datasets/paraschiv/cluj-raw-datasets/rdd2022', 'data/datasets/rdd2022', dirs_exist_ok=True)

print("Copying Pothole-600...")
# Applying the same path structure to pothole600
shutil.copytree('/kaggle/input/datasets/paraschiv/cluj-raw-datasets/pothole600', 'data/datasets/pothole600', dirs_exist_ok=True)

print("Data successfully copied to writable space! Folder tree matches local setup.")

Copying RDD2022 (This will take a few minutes for 28k images)...
Copying Pothole-600...
Data successfully copied to writable space! Folder tree matches local setup.


In [3]:
# Move all contents from the nested rdd2022 folder up one level
!mv data/datasets/rdd2022/rdd2022/* data/datasets/rdd2022/

# Remove the empty nested folder to clean things up
!rm -rf data/datasets/rdd2022/rdd2022

print("RDD2022 folder flattened successfully!")

RDD2022 folder flattened successfully!


In [4]:
!mv data/datasets/pothole600/pothole600/* data/datasets/pothole600/
!rm -rf data/datasets/pothole600/pothole600

In [5]:
!python ml/detection/data_prep/prep_rdd2022.py --dataset_dir data/datasets/rdd2022
!python ml/detection/data_prep/prep_pothole600.py
!python ml/detection/data_prep/merge_datasets.py
!python ml/detection/data_prep/coco_to_yolo.py

2026-03-16 11:50:26.499 | INFO     | __main__:main:130 - ── RDD2022 -> COCO conversion ──────────────────────────────
2026-03-16 11:50:26.500 | INFO     | __main__:main:153 - Processing: train/
  train: 100%|██████████████████████████| 26869/26869 [00:06<00:00, 4060.60it/s]
2026-03-16 11:50:33.477 | INFO     | __main__:build_coco:125 -   missing labels: 0  skipped: 0
2026-03-16 11:50:34.179 | SUCCESS  | __main__:main:157 -   [train] 26869 images, 41667 annotations -> annotations_train.json
2026-03-16 11:50:34.179 | INFO     | __main__:main:153 - Processing: val/
  val: 100%|███████████████████████████████| 5758/5758 [00:08<00:00, 693.34it/s]
2026-03-16 11:50:42.540 | INFO     | __main__:build_coco:125 -   missing labels: 0  skipped: 0
2026-03-16 11:50:42.676 | SUCCESS  | __main__:main:157 -   [val] 5758 images, 8776 annotations -> annotations_val.json
2026-03-16 11:50:42.677 | INFO     | __main__:main:153 - Processing: test/
  test: 100%|█████████████████████████████| 5758/5758 [00:05<

In [6]:
# Nuke the loggers so they don't crash the end of the epoch
!pip uninstall -y wandb ray

# Copy your newest weights and pretend they are the base model
!cp "/kaggle/input/datasets/paraschiv/cluj-road-weights-51-75/last (1).pt" ./rtdetr-l.pt

# Start training for the next 18 epochs to safely fit the 12-hour window!
!python ml/detection/train.py --freeze_epochs 0 --epochs 18 --device 0 --workers 4

Found existing installation: wandb 0.24.0
Uninstalling wandb-0.24.0:
  Successfully uninstalled wandb-0.24.0
Found existing installation: ray 2.53.0
Uninstalling ray-2.53.0:
  Successfully uninstalled ray-2.53.0

[YAML] dataset.yaml OK → /kaggle/working/Cluj-Road-Intelligence-System/data/detection/dataset.yaml

── Environment ───────────────────────────────────────────────
  PyTorch  : 2.2.2+cu121
  CUDA     : 12.1
  GPU      : Tesla P100-PCIE-16GB
  VRAM     : 15.9 GB

[PSO] pso_best.json not found — using defaults.
      Run ml/optimization/pso_hyperparams.py first for optimised training.

── Training configuration ────────────────────────────────────
  Pretrained    : /kaggle/working/Cluj-Road-Intelligence-System/rtdetr-l.pt
  Dataset       : /kaggle/working/Cluj-Road-Intelligence-System/data/detection/dataset.yaml
  Total epochs  : 18  (freeze 0 + fine-tune 18)
  Batch / GPU   : 16   (effective: 64 with grad_accumulate=4)
  Image size    : 640×640
  LR phase 1    : 0.0001
  LR phas

In [7]:
import shutil
import os

print("Zipping the runs folder...")
shutil.make_archive('/kaggle/working/runs_backup', 'zip', 'runs')
print("Successfully zipped! Ready for download.")

Zipping the runs folder...
Successfully zipped! Ready for download.
